## LSTM project
#### In this prject we build a model that can predict next word of the sentence

In [12]:
import pandas as pd

df = pd.read_csv("movie_lines.csv")

# Rename the existing 5 columns
df.columns = [
    "line_id",
    "character_id",
    "movie_id",
    "character",
    "text"
]

print(df.head())
print(df.columns)

  line_id character_id movie_id character         text
0   L1044           u2       m0   CAMERON  They do to!
1    L985           u0       m0    BIANCA   I hope so.
2    L984           u2       m0   CAMERON    She okay?
3    L925           u0       m0    BIANCA    Let's go.
4    L924           u2       m0   CAMERON          Wow
Index(['line_id', 'character_id', 'movie_id', 'character', 'text'], dtype='object')


In [14]:
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [16]:
print(df.columns.tolist())


['L1045', 'u0', 'm0', 'BIANCA', 'They do not!']


In [17]:
df.head()

,L1045,u0,m0,BIANCA,They do not!
0,L1044,u2,m0,CAMERON,They do to!
1,L985,u0,m0,BIANCA,I hope so.
2,L984,u2,m0,CAMERON,She okay?
3,L925,u0,m0,BIANCA,Let's go.
4,L924,u2,m0,CAMERON,Wow


In [20]:
import pandas as pd

# Load CSV
df = pd.read_csv("movie_lines.csv")

# Force correct column names
df.columns = [
    "line_id",
    "character_id",
    "movie_id",
    "character",
    "text"
]

# Verify
print(df.columns.tolist())
print(df.head())

# Create document
document = " ".join(
    df["text"]
    .dropna()
    .astype(str)
    .tolist()
)

print("\nDocument created successfully!")
print("Total characters:", len(document))
print("\nFirst 500 characters:")
print(document[:500])

['line_id', 'character_id', 'movie_id', 'character', 'text']
  line_id character_id movie_id character         text
0   L1044           u2       m0   CAMERON  They do to!
1    L985           u0       m0    BIANCA   I hope so.
2    L984           u2       m0   CAMERON    She okay?
3    L925           u0       m0    BIANCA    Let's go.
4    L924           u2       m0   CAMERON          Wow

Document created successfully!
Total characters: 15708630

First 500 characters:
They do to! I hope so. She okay? Let's go. Wow Okay -- you're gonna need to learn how to lie. No Like my fear of wearing pastels? What good stuff? I figured you'd get to the good stuff eventually. Thank God!  If I had to hear one more story about your coiffure... Me.  This endless ...blonde babble. I'm like boring myself. What crap? do you listen to this crap? No... You always been this selfish? But Then that's all you had to say. Well no... You never wanted to go out with 'me did you? I was? To


In [21]:
# tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kv997\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kv997\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [22]:
# tokenize
tokens = word_tokenize(document.lower())

In [23]:
# vocab
vocab = {'<UNK>':0}

for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab)
vocab

{'<UNK>': 0,
 'they': 1,
 'do': 2,
 'to': 3,
 '!': 4,
 'i': 5,
 'hope': 6,
 'so': 7,
 '.': 8,
 'she': 9,
 'okay': 10,
 '?': 11,
 'let': 12,
 "'s": 13,
 'go': 14,
 'wow': 15,
 '--': 16,
 'you': 17,
 "'re": 18,
 'gon': 19,
 'na': 20,
 'need': 21,
 'learn': 22,
 'how': 23,
 'lie': 24,
 'no': 25,
 'like': 26,
 'my': 27,
 'fear': 28,
 'of': 29,
 'wearing': 30,
 'pastels': 31,
 'what': 32,
 'good': 33,
 'stuff': 34,
 'figured': 35,
 "'d": 36,
 'get': 37,
 'the': 38,
 'eventually': 39,
 'thank': 40,
 'god': 41,
 'if': 42,
 'had': 43,
 'hear': 44,
 'one': 45,
 'more': 46,
 'story': 47,
 'about': 48,
 'your': 49,
 'coiffure': 50,
 '...': 51,
 'me': 52,
 'this': 53,
 'endless': 54,
 'blonde': 55,
 'babble': 56,
 "'m": 57,
 'boring': 58,
 'myself': 59,
 'crap': 60,
 'listen': 61,
 'always': 62,
 'been': 63,
 'selfish': 64,
 'but': 65,
 'then': 66,
 'that': 67,
 'all': 68,
 'say': 69,
 'well': 70,
 'never': 71,
 'wanted': 72,
 'out': 73,
 'with': 74,
 "'me": 75,
 'did': 76,
 'was': 77,
 'tons': 78

In [24]:
len(vocab)

56005

In [12]:
input_sentences = document.split('\n')

In [13]:
input_sentences

['About the Program',
 'What is the course fee for  Data Science Mentorship Program (DSMP 2023)',
 'The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.',
 'What is the total duration of the course?',
 'The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)',
 'What is the syllabus of the mentorship program?',
 'We will be covering the following modules:',
 'Python Fundamentals',
 'Python libraries for Data Science',
 'Data Analysis',
 'SQL for Data Science',
 'Maths for Machine Learning',
 'ML Algorithms',
 'Practical ML',
 'MLOPs',
 'Case studies',
 'You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390',
 'Will Deep Learning and NLP be a part of this program?',
 'No, NLP and Deep Learning both are not a part of this program’s curriculum.',
 'What if I miss a live session? Will I get a recording 

In [14]:
def txt_to_num(sentence, vocab):
    num_sentence = []
    for token in sentence:
        if token in vocab:
            num_sentence.append(vocab[token])
        else :
            num_sentence.append(vocab['<UNK>'])

    return num_sentence


In [15]:
in_num_sentence = []

for sentence in input_sentences:

    in_num_sentence.append(txt_to_num(word_tokenize(sentence.lower()), vocab))

In [16]:
len(in_num_sentence)

78

In [17]:
training_sequence = []

for sentence in in_num_sentence:

    for i in range(1, len(sentence)):

        training_sequence.append(sentence[:i+1])

In [18]:
  # sincce all the words are not of same size so we have to do padding
len_list=[]
for sequence in training_sequence:
    len_list.append(len(sequence))

max(len_list)

62

In [19]:
padded_training_seq =[]

for sequence in training_sequence:

    padded_training_seq.append([0]* (max(len_list)- len(sequence)) + sequence)
padded_training_seq =torch.tensor(padded_training_seq, dtype = torch.long)

In [20]:
padded_training_seq.shape

torch.Size([942, 62])

In [21]:
 # har row ka last element output hai 

x = padded_training_seq[:,:-1]
y = padded_training_seq[:,-1]

#### Now we have to build a dataset and dataloader 

In [22]:
class customdataset (Dataset):

    def __init__(self, x,y):
        self.x = x
        self.y = y
    def __len__(self):
        return self.x.shape[0]
    
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]

In [23]:
dataset=  customdataset(x,y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

#### LSTM Architecture

In [24]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [25]:
model = LSTMModel(len(vocab))

In [26]:
# bring it to gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

LSTMModel(
  (embedding): Embedding(289, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=289, bias=True)
)

In [27]:
learning_rate = 0.001
epochs  = 50

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [28]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 166.1180
Epoch: 2, Loss: 146.8525
Epoch: 3, Loss: 133.6588
Epoch: 4, Loss: 120.8611
Epoch: 5, Loss: 108.7010
Epoch: 6, Loss: 97.3151
Epoch: 7, Loss: 86.5699
Epoch: 8, Loss: 76.6961
Epoch: 9, Loss: 67.6975
Epoch: 10, Loss: 59.5455
Epoch: 11, Loss: 52.3152
Epoch: 12, Loss: 45.9333
Epoch: 13, Loss: 40.3246
Epoch: 14, Loss: 35.4291
Epoch: 15, Loss: 30.9405
Epoch: 16, Loss: 27.1243
Epoch: 17, Loss: 23.9687
Epoch: 18, Loss: 21.2470
Epoch: 19, Loss: 18.9019
Epoch: 20, Loss: 16.8062
Epoch: 21, Loss: 15.2750
Epoch: 22, Loss: 13.6794
Epoch: 23, Loss: 12.7252
Epoch: 24, Loss: 11.5403
Epoch: 25, Loss: 10.5247
Epoch: 26, Loss: 9.9759
Epoch: 27, Loss: 9.0721
Epoch: 28, Loss: 8.5713
Epoch: 29, Loss: 8.0358
Epoch: 30, Loss: 7.7618
Epoch: 31, Loss: 7.3401
Epoch: 32, Loss: 6.9376
Epoch: 33, Loss: 6.6861
Epoch: 34, Loss: 6.4297
Epoch: 35, Loss: 6.1584
Epoch: 36, Loss: 5.8732
Epoch: 37, Loss: 5.7939
Epoch: 38, Loss: 5.6190
Epoch: 39, Loss: 5.4582
Epoch: 40, Loss: 5.2764
Epoch: 41, Loss: 5.

In [43]:
# prediction 

def prediction (model, vocab, text):
    # tokenize 
    tokenize_text = word_tokenize(text.lower())

    # convert to numbers 
    num_text =txt_to_num(tokenize_text, vocab)

    # padding
    padd_txt = torch.tensor([0] * (61 - len(num_text)) + num_text, dtype=torch.long, device=device).unsqueeze(0)
    # senf to model
    output = model(padd_txt)

    # predicted index
    value , index = torch.max(output, dim=1)

    # merge with text 
    return text + " " + list(vocab.keys())[index]

In [45]:
prediction(model, vocab, "The course follows")

'The course follows a'

In [48]:
import time

num_tokens = 10
input_text = "you have to learn"

for i in range(num_tokens):
    output_text = prediction(model, vocab, input_text)
    print(output_text)
    input_text = output_text

    time.sleep(0.2)

you have to learn all
you have to learn all the
you have to learn all the course
you have to learn all the course assessments
you have to learn all the course assessments .
you have to learn all the course assessments . so
you have to learn all the course assessments . so we
you have to learn all the course assessments . so we dont
you have to learn all the course assessments . so we dont guarantee
you have to learn all the course assessments . so we dont guarantee you
